In [5]:
grid=[['X',8,4,'X','X',11,'X','X','X',5,'X'],
      ['X','X','X','X','X','X','X',11,'X','X','X'],
      ['X','X',7,'X','X',1,'X','X','X','X',14],
      [28,'X','X',51,'X','X',1,'X',6,'X','X'],
      ['X',22,'X','X','X','X','X','X',4,'X',1],
      ['X','X','X',15,'X',10,'X',0,'X','X','X'],
      [11,'X',11,'X','X','X','X','X','X',9,'X'],
      ['X','X',17,'X',14,'X','X',10,'X','X',13],
      [30,'X','X','X','X',6,'X','X',45,'X','X'],
      ['X','X','X',10,'X','X','X','X','X','X','X'],
      ['X',26,'X','X','X',0,'X','X',77,61,'X']]
len(grid[0])
print(len(grid))

for g in grid:
    if len(g)!=11:
        print('uhoh')

11


In [10]:
from functools import lru_cache
from math import ceil

grid = [
    ['X',8,4,'X','X',11,'X','X','X',5,'X'],
    ['X','X','X','X','X','X','X',11,'X','X','X'],
    ['X','X',7,'X','X',1,'X','X','X','X',14],
    [28,'X','X',51,'X','X',1,'X',6,'X','X'],
    ['X',22,'X','X','X','X','X','X',4,'X',1],
    ['X','X','X',15,'X',10,'X',0,'X','X','X'],
    [11,'X',11,'X','X','X','X','X','X',9,'X'],
    ['X','X',17,'X',14,'X','X',10,'X','X',13],
    [30,'X','X','X','X',6,'X','X',45,'X','X'],
    ['X','X','X',10,'X','X','X','X','X','X','X'],
    ['X',26,'X','X','X',0,'X','X',77,61,'X']
]

ROWS = len(grid)
COLS = len(grid[0])
N = ROWS * COLS   # 121


# ============================================================
# 1. Extract observed clues
# ============================================================

clues = []

for r in range(ROWS):
    for c in range(COLS):
        if grid[r][c] != 'X':
            clues.append((r, c, int(grid[r][c])))

print("Observed clues:", len(clues))


# ============================================================
# 2. Derive size-only metric bounds
#
# For clues x,y:
#
# |f(x)-f(y)| <= d(x,y)
#
# A Manhattan path of length L exists.
#
# If M is the largest state size, then every edge cost <= M,
# so that particular path costs at most L*M.
#
# Hence:
#
# |f(x)-f(y)| <= L*M
#
# and therefore
#
# M >= ceil(|f(x)-f(y)| / L)
# ============================================================

bounds = []

for i in range(len(clues)):
    r1, c1, f1 = clues[i]

    for j in range(i + 1, len(clues)):
        r2, c2, f2 = clues[j]

        L = abs(r1 - r2) + abs(c1 - c2)

        if L == 0:
            continue

        gap = abs(f1 - f2)

        required_M = ceil(gap / L)

        bounds.append(
            (
                required_M,
                gap,
                L,
                (r1, c1, f1),
                (r2, c2, f2),
            )
        )

bounds.sort(reverse=True)

strongest = bounds[0]
MIN_MAX_STATE = strongest[0]

print("\nStrongest size-only metric bound:")
print("M >=", MIN_MAX_STATE)
print("coming from:")
print(strongest[3], "<->", strongest[4])
print("|Δf| =", strongest[1])
print("grid distance =", strongest[2])


# ============================================================
# 3. Dynamic programming:
#
# Count integer partitions of `total`
# into exactly `k` positive parts,
# with every part <= max_part.
#
# This counts unordered state-size multisets.
# ============================================================

@lru_cache(maxsize=None)
def count_partitions(total, max_part, k):
    """
    Number of unordered partitions of `total`
    into exactly `k` positive integers,
    each <= max_part.
    """

    if total == 0:
        return 1 if k == 0 else 0

    if total < 0:
        return 0

    if max_part <= 0:
        return 0

    if k <= 0:
        return 0

    # Impossible to fit k positive parts
    if total < k:
        return 0

    # Even k copies of max_part cannot reach total
    if total > k * max_part:
        return 0

    # Either:
    #
    # 1. don't use max_part
    # 2. use one max_part and continue
    #
    return (
        count_partitions(total, max_part - 1, k)
        +
        count_partitions(total - max_part, max_part, k - 1)
    )


# ============================================================
# 4. Count spectra satisfying largest-state >= MIN_MAX_STATE
#
# Count:
#
# all partitions of 121 into k parts
# minus
# partitions whose every part <= MIN_MAX_STATE-1
# ============================================================

def count_with_large_state(k):
    all_partitions = count_partitions(
        N,
        N,
        k
    )

    all_small = count_partitions(
        N,
        MIN_MAX_STATE - 1,
        k
    )

    return all_partitions - all_small


# ============================================================
# 5. Lower bound on state count from observed zero clues
#
# With positive weights:
# f(x)=0 iff x is a capitol.
#
# Two observed zeros therefore certify at least two capitols,
# hence at least two states if there is one capitol/state.
# ============================================================

zero_clues = [
    (r, c)
    for r, c, value in clues
    if value == 0
]

MIN_STATES = len(zero_clues)

print("\nObserved zero clues:", zero_clues)
print("Minimum state count:", MIN_STATES)


# ============================================================
# 6. Count candidate size spectra by number of states
# ============================================================

counts_by_k = {}

for k in range(MIN_STATES, N + 1):

    count = count_with_large_state(k)

    if count > 0:
        counts_by_k[k] = count


total_candidates = sum(counts_by_k.values())


print("\n==========================================")
print("SIZE-SPECTRUM CANDIDATES")
print("==========================================")

print(
    "Total candidate state-size multisets:",
    f"{total_candidates:,}"
)

print("\nCandidates by number of states:")

for k, count in counts_by_k.items():
    print(
        f"{k:3d} states : "
        f"{count:12,d}"
    )


# ============================================================
# 7. Find most combinatorially ambiguous state count
# ============================================================

peak_k = max(
    counts_by_k,
    key=counts_by_k.get
)

print("\nPeak ambiguity:")

print(
    peak_k,
    "states ->",
    f"{counts_by_k[peak_k]:,}",
    "size spectra"
)


# ============================================================
# 8. Top metric bounds
# ============================================================

print("\n==========================================")
print("TOP METRIC SIZE BOUNDS")
print("==========================================")

for required_M, gap, L, a, b in bounds[:15]:

    print(
        f"M >= {required_M:2d} | "
        f"gap={gap:2d} | "
        f"steps={L:2d} | "
        f"{a} <-> {b}"
    )

Observed clues: 33

Strongest size-only metric bound:
M >= 26
coming from:
(10, 5, 0) <-> (10, 8, 77)
|Δf| = 77
grid distance = 3

Observed zero clues: [(5, 7), (10, 5)]
Minimum state count: 2

SIZE-SPECTRUM CANDIDATES
Total candidate state-size multisets: 798,982,326

Candidates by number of states:
  2 states :           60
  3 states :        1,220
  4 states :       12,600
  5 states :       80,626
  6 states :      357,534
  7 states :    1,180,186
  8 states :    3,051,735
  9 states :    6,450,527
 10 states :   11,557,764
 11 states :   18,101,960
 12 states :   25,418,252
 13 states :   32,666,365
 14 states :   39,068,309
 15 states :   44,070,813
 16 states :   47,399,087
 17 states :   49,030,433
 18 states :   49,123,454
 19 states :   47,941,479
 20 states :   45,786,228
 21 states :   42,952,560
 22 states :   39,700,807
 23 states :   36,245,158
 24 states :   32,751,070
 25 states :   29,339,449
 26 states :   26,092,945
 27 states :   23,063,597
 28 states :   20,279,

In [13]:
import itertools
import numpy as np

from collections import defaultdict
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix


# ============================================================
# INPUT
# ============================================================

grid = [
    ['X',8,4,'X','X',11,'X','X','X',5,'X'],
    ['X','X','X','X','X','X','X',11,'X','X','X'],
    ['X','X',7,'X','X',1,'X','X','X','X',14],
    [28,'X','X',51,'X','X',1,'X',6,'X','X'],
    ['X',22,'X','X','X','X','X','X',4,'X',1],
    ['X','X','X',15,'X',10,'X',0,'X','X','X'],
    [11,'X',11,'X','X','X','X','X','X',9,'X'],
    ['X','X',17,'X',14,'X','X',10,'X','X',13],
    [30,'X','X','X','X',6,'X','X',45,'X','X'],
    ['X','X','X',10,'X','X','X','X','X','X','X'],
    ['X',26,'X','X','X',0,'X','X',77,61,'X']
]

ROWS = len(grid)
COLS = len(grid[0])

# Increase later if desired.
#
# For L <= 6, number of Manhattan shortest paths is tiny:
# max C(6,3) = 20.
MAX_PATH_LENGTH = 6

# Actual state sizes are integers.
USE_INTEGER_WEIGHTS = True

MIN_EDGE_WEIGHT = 1
MAX_EDGE_WEIGHT = ROWS * COLS   # 121


# ============================================================
# 1. BUILD GRID GRAPH
# ============================================================

def canonical_edge(a, b):
    return tuple(sorted((a, b)))


edges = []
adj = defaultdict(list)

for r in range(ROWS):
    for c in range(COLS):

        u = (r, c)

        if r + 1 < ROWS:
            v = (r + 1, c)

            e = canonical_edge(u, v)

            edges.append(e)
            adj[u].append(v)
            adj[v].append(u)

        if c + 1 < COLS:
            v = (r, c + 1)

            e = canonical_edge(u, v)

            edges.append(e)
            adj[u].append(v)
            adj[v].append(u)


edge_to_idx = {
    e: i
    for i, e in enumerate(edges)
}

E = len(edges)

print("Vertices:", ROWS * COLS)
print("Edges:", E)


# ============================================================
# 2. EXTRACT CLUES
# ============================================================

clues = []

for r in range(ROWS):
    for c in range(COLS):

        if grid[r][c] != 'X':
            clues.append(
                ((r, c), int(grid[r][c]))
            )

print("Observed clues:", len(clues))


# ============================================================
# 3. GENERATE ALL SHORTEST MANHATTAN PATHS
#
# We only do this for clue pairs <= MAX_PATH_LENGTH apart.
#
# Every one of these paths gives:
#
#       sum(edge weights) >= |f(x)-f(y)|
#
# ============================================================

def shortest_manhattan_paths(start, goal):

    r1, c1 = start
    r2, c2 = goal

    dr = r2 - r1
    dc = c2 - c1

    vertical_step = (
        (1, 0) if dr > 0
        else (-1, 0)
    )

    horizontal_step = (
        (0, 1) if dc > 0
        else (0, -1)
    )

    vertical_count = abs(dr)
    horizontal_count = abs(dc)

    steps = (
        ["V"] * vertical_count
        +
        ["H"] * horizontal_count
    )

    # Unique permutations only.
    #
    # For length <= 6 this is tiny.
    permutations = set(
        itertools.permutations(steps)
    )

    for perm in permutations:

        current = start
        path_edges = []

        for direction in perm:

            r, c = current

            if direction == "V":
                step = vertical_step
            else:
                step = horizontal_step

            nxt = (
                r + step[0],
                c + step[1]
            )

            path_edges.append(
                canonical_edge(current, nxt)
            )

            current = nxt

        yield tuple(path_edges)


# ============================================================
# 4. BUILD PATH INEQUALITIES
#
# Deduplicate identical edge sets.
#
# If the same path receives multiple lower bounds,
# retain the strongest one.
# ============================================================

cuts = {}

pair_stats = []

for i in range(len(clues)):

    x, fx = clues[i]

    for j in range(i + 1, len(clues)):

        y, fy = clues[j]

        L = (
            abs(x[0] - y[0])
            +
            abs(x[1] - y[1])
        )

        if L == 0 or L > MAX_PATH_LENGTH:
            continue

        gap = abs(fx - fy)

        if gap == 0:
            continue

        paths = list(
            shortest_manhattan_paths(x, y)
        )

        pair_stats.append(
            (gap, L, x, fx, y, fy, len(paths))
        )

        for path in paths:

            # A shortest Manhattan path cannot repeat an edge,
            # so frozenset is fine here.
            key = frozenset(path)

            cuts[key] = max(
                cuts.get(key, 0),
                gap
            )


print()
print("Clue pairs used:", len(pair_stats))
print("Unique path inequalities:", len(cuts))


# ============================================================
# 5. CONVERT CUTS TO SPARSE LINEAR CONSTRAINT MATRIX
#
#       A w >= b
# ============================================================

num_constraints = len(cuts)

A = lil_matrix(
    (num_constraints, E),
    dtype=float
)

b = np.zeros(num_constraints)

for row, (path_edges, rhs) in enumerate(cuts.items()):

    for edge in path_edges:
        A[row, edge_to_idx[edge]] = 1.0

    b[row] = rhs

A = A.tocsr()

constraints = LinearConstraint(
    A,
    lb=b,
    ub=np.full(num_constraints, np.inf)
)

bounds = Bounds(
    lb=np.full(E, MIN_EDGE_WEIGHT),
    ub=np.full(E, MAX_EDGE_WEIGHT)
)

if USE_INTEGER_WEIGHTS:
    integrality = np.ones(E, dtype=int)
else:
    integrality = None


# ============================================================
# 6. SANITY CHECK: FIND ANY FEASIBLE EDGE ASSIGNMENT
# ============================================================

print()
print("Checking relaxed metric feasibility...")

result = milp(
    c=np.zeros(E),
    integrality=integrality,
    bounds=bounds,
    constraints=constraints,
    options={
        "disp": False
    }
)

if not result.success:

    raise RuntimeError(
        "Metric relaxation is infeasible!\n"
        + result.message
    )

print("Feasible.")


# ============================================================
# 7. MINIMIZE EACH EDGE INDIVIDUALLY
#
# This gives:
#
#   lower[e] =
#       smallest value edge e can take
#       while satisfying ALL included path inequalities.
#
# Therefore any larger value is not assumed.
# ============================================================

edge_lower = np.ones(E, dtype=int)

print()
print("Computing certified edge lower bounds...")

for edge_idx in range(E):

    objective = np.zeros(E)
    objective[edge_idx] = 1.0

    result = milp(
        c=objective,
        integrality=integrality,
        bounds=bounds,
        constraints=constraints,
        options={
            "disp": False
        }
    )

    if not result.success:
        raise RuntimeError(
            f"Optimization failed for edge {edge_idx}"
        )

    edge_lower[edge_idx] = int(
        round(result.fun)
    )

    if (
        (edge_idx + 1) % 20 == 0
        or edge_idx == E - 1
    ):
        print(
            f"  solved {edge_idx + 1:3d}/{E}"
        )


# ============================================================
# 8. CELL-LEVEL MINIMUM STATE SIZE
#
# If
#
#       w(u,v) >= L
#
# and
#
#       w(u,v) = min(size(u), size(v))
#
# then
#
#       size(u) >= L
#       size(v) >= L
#
# So for each cell:
#
#       minimum possible state size
#       =
#       max lower bound over incident edges
# ============================================================

cell_min_state = np.ones(
    (ROWS, COLS),
    dtype=int
)

for edge, idx in edge_to_idx.items():

    u, v = edge

    L = edge_lower[idx]

    cell_min_state[u] = max(
        cell_min_state[u],
        L
    )

    cell_min_state[v] = max(
        cell_min_state[v],
        L
    )


# ============================================================
# 9. REPORT NONTRIVIAL EDGE BOUNDS
# ============================================================

interesting_edges = []

for edge, idx in edge_to_idx.items():

    L = edge_lower[idx]

    if L > 1:
        interesting_edges.append(
            (L, edge)
        )

interesting_edges.sort(
    reverse=True
)

print()
print("=" * 70)
print("NONTRIVIAL EDGE LOWER BOUNDS")
print("=" * 70)

print(
    "Number of edges with lower bound > 1:",
    len(interesting_edges)
)

for L, edge in interesting_edges:

    print(
        f"{edge[0]} <-> {edge[1]}"
        f"     w >= {L}"
    )


# ============================================================
# 10. PRINT CELL MINIMUM-STATE-SIZE MAP
# ============================================================

print()
print("=" * 70)
print("MINIMUM STATE SIZE REQUIRED AT EACH CELL")
print("=" * 70)

for r in range(ROWS):

    print(
        " ".join(
            f"{cell_min_state[r,c]:3d}"
            for c in range(COLS)
        )
    )


# ============================================================
# 11. HISTOGRAM
# ============================================================

values, counts = np.unique(
    cell_min_state,
    return_counts=True
)

print()
print("=" * 70)
print("CELL REQUIREMENT HISTOGRAM")
print("=" * 70)

for value, count in zip(values, counts):

    print(
        f"state size >= {value:3d}: "
        f"{count:3d} cells"
    )


# ============================================================
# 12. STRONGEST CLUE-PAIR CONSTRAINTS USED
# ============================================================

pair_stats.sort(
    reverse=True
)

print()
print("=" * 70)
print("STRONGEST SHORT-PATH CLUE CONSTRAINTS")
print("=" * 70)

for (
    gap,
    length,
    x,
    fx,
    y,
    fy,
    num_paths
) in pair_stats[:25]:

    print(
        f"|Δf|={gap:3d}   "
        f"L={length}   "
        f"paths={num_paths:3d}   "
        f"{x}:{fx} <-> {y}:{fy}"
    )

Vertices: 121
Edges: 220
Observed clues: 33

Clue pairs used: 222
Unique path inequalities: 1252

Checking relaxed metric feasibility...
Feasible.

Computing certified edge lower bounds...
  solved  20/220
  solved  40/220
  solved  60/220
  solved  80/220
  solved 100/220
  solved 120/220
  solved 140/220
  solved 160/220
  solved 180/220
  solved 200/220
  solved 220/220

NONTRIVIAL EDGE LOWER BOUNDS
Number of edges with lower bound > 1: 4
(10, 8) <-> (10, 9)     w >= 16
(6, 2) <-> (7, 2)     w >= 6
(0, 1) <-> (0, 2)     w >= 4
(3, 8) <-> (4, 8)     w >= 2

MINIMUM STATE SIZE REQUIRED AT EACH CELL
  1   4   4   1   1   1   1   1   1   1   1
  1   1   1   1   1   1   1   1   1   1   1
  1   1   1   1   1   1   1   1   1   1   1
  1   1   1   1   1   1   1   1   2   1   1
  1   1   1   1   1   1   1   1   2   1   1
  1   1   1   1   1   1   1   1   1   1   1
  1   1   6   1   1   1   1   1   1   1   1
  1   1   6   1   1   1   1   1   1   1   1
  1   1   1   1   1   1   1   1   1   1  